# ۱. شروع پروژه در Kaggle و آماده‌سازی داده

این فایل، نقطهٔ شروع است. به GPU برای آماده‌سازی داده نیاز ندارید. Internet را روشن کنید و سلول‌ها را از بالا به پایین اجرا کنید.
این نوت‌بوک وزن مدل را آموزش نمی‌دهد. خروجی آن دادهٔ گفتاری دوصدایی و فایل‌های توضیح داده برای مراحل بعدی است.

مسیر اصلی: **این فایل → 12_surt_streaming_inference → آموزش اختیاری 13 → ارزیابی 14**.
نوت‌بوک 11 (WhisperRT) فقط مقایسهٔ اختیاری است. VAD، OSD و مسیریاب جزو مسیر اصلی SURT نیستند.

پیش‌نیاز مهم: نسخهٔ اصلاح‌شدهٔ سورس باید ابتدا در GitHub شما قرار گرفته باشد؛ clone تغییراتِ فقط موجود روی کامپیوتر شخصی را نمی‌بیند.
راهنمای کامل فارسی: `docs/KAGGLE_WORKFLOW_FA.md`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/manzourali/English-Streaming-ASR.git"
REPOSITORY_REF = "master"
ROOT = Path("/kaggle/working/English-Streaming-ASR")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(ROOT)], check=True)
os.chdir(ROOT)
if not (ROOT / "scripts/check_surt_assets.py").is_file():
    raise RuntimeError("This GitHub checkout predates the Kaggle workflow update. Publish the updated source, then start a fresh session.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[kaggle,dev]"], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
ARTIFACTS = Path("/kaggle/working/asr_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str, arguments)], check=True)

## بررسی محیط

این تست‌ها اجرای مدل واقعی نیستند. نصب پایه، PyTorch/CUDA پیش‌فرض Kaggle را تعویض نمی‌کند.
برای اجرای مدل در مرحلهٔ 12 باید GPU را از تنظیمات Kaggle فعال کنید.

In [ ]:
import platform
print("Python:", platform.python_version())
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda, "GPU:", torch.cuda.is_available())
except ImportError:
    print("PyTorch is absent; choose a Kaggle GPU image before model inference.")
run_script("smoke_test.py", "--phase", "0")
run_script("validate_notebooks.py")
subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "not integration"], check=True)

## انتخاب اندازهٔ آزمایش

`pilot` فقط dev-clean واقعی را دانلود می‌کند و گویندگان آن را به سه گروه جدا برای تمرین/اعتبارسنجی/آزمایش تقسیم می‌کند؛ این **بنچمارک رسمی test-clean نیست**.
`standard` به train-clean-100، dev-clean و test-clean نیاز دارد و دانلود بزرگ‌تری دارد. ابتدا pilot را اجرا کنید.
اگر LibriSpeech را با Add Input متصل کرده‌اید، `SOURCE_ROOT` را پوشه‌ای قرار دهید که مستقیماً `dev-clean` یا سه پوشهٔ رسمی را دارد.
از اجرای دوبارهٔ سلول‌های تولید روی دادهٔ قبلی جلوگیری می‌شود؛ برای آزمایش جدید، پوشهٔ خروجی تازه انتخاب کنید.

In [ ]:
LAYOUT = "pilot"
SOURCE_ROOT = None
DOWNLOAD_IF_MISSING = True
MAX_SOURCE_UTTERANCES_PER_SPLIT = 100
MIXTURES_PER_SPLIT = 20
DATA_ROOT = ARTIFACTS / "data" / f"{LAYOUT}_v1"
SOURCE_MANIFESTS = ARTIFACTS / "source_manifests" / LAYOUT

if SOURCE_ROOT is None:
    if not DOWNLOAD_IF_MISSING:
        raise RuntimeError("Attach LibriSpeech and set SOURCE_ROOT, or enable DOWNLOAD_IF_MISSING.")
    download_root = Path("/kaggle/working/librispeech_sources")
    run_script("download_librispeech.py", "--layout", LAYOUT, "--output-root", download_root)
    SOURCE_ROOT = download_root / "LibriSpeech"
else:
    SOURCE_ROOT = Path(SOURCE_ROOT)
print("Speech source:", SOURCE_ROOT)

In [ ]:
run_script("prepare_dataset.py", "--source-root", SOURCE_ROOT,
    "--output-root", SOURCE_MANIFESTS, "--layout", LAYOUT,
    "--max-per-split", MAX_SOURCE_UTTERANCES_PER_SPLIT)
print((SOURCE_MANIFESTS / "preparation.json").read_text())

## ساخت گفتار دوصدایی و بررسی جدایی گویندگان

اینجا از صدای واقعی استفاده می‌شود؛ گزینهٔ `--demo` عمداً اجرا نمی‌شود، چون demo فقط موج سینوسی با متن ساختگی می‌سازد.
`control` شامل دو قطعهٔ غیرهم‌زمان است و معادل آزمایش تک‌گویندهٔ پاک نیست.
تعداد 20 نمونه در هر بخش فقط برای اجرای آزمایشی کوچک است.

In [ ]:
import yaml
if DATA_ROOT.exists():
    raise RuntimeError(f"Dataset already exists: {DATA_ROOT}. Reuse it or choose a new versioned output path.")
data_override = ARTIFACTS / "generation.yaml"
data_override.write_text(yaml.safe_dump({
    "data": {"backend": "local", "manifest_root": str(SOURCE_MANIFESTS), "dataset": "local_librispeech"},
    "generation": {"num_samples": MIXTURES_PER_SPLIT, "max_source_samples": MAX_SOURCE_UTTERANCES_PER_SPLIT},
    "output": {"root": str(DATA_ROOT)},
    "paths": {"outputs": str(ARTIFACTS / "preparation_outputs")}
}, sort_keys=False))
run_script("generate_overlap.py", "--override", data_override)


In [ ]:
from streaming_asr.datasets.manifests import read_overlap_manifest
from streaming_asr.datasets.validators import validate_overlap_records, check_speaker_disjointness
splits = {name: read_overlap_manifest(DATA_ROOT / "manifests" / f"{name}.jsonl")
          for name in ("train", "validation", "test")}
errors = check_speaker_disjointness(splits)
for records in splits.values():
    errors.extend(validate_overlap_records(records, check_source_files=False))
assert not errors, errors
print("Validated mixtures:", {name: len(records) for name, records in splits.items()})
print("Dataset metadata:", (DATA_ROOT / "dataset_info.json").read_text())

## ذخیرهٔ خروجی و گام بعد

۱. خروجی پوشهٔ `/kaggle/working/asr_artifacts` را با **Save Version** همراه با خروجی‌ها ذخیره کنید.
۲. نوت‌بوک `notebooks/12_surt_streaming_inference.ipynb` را باز کنید؛ برای دانلود مدل GPU لازم نیست؛ فایل را می‌توانید از مخزن وارد Kaggle کنید. سلول‌های این فایل، نوت‌بوک بعدی را خودکار اجرا نمی‌کنند.
۳. در جلسهٔ تازه با **Add Input** خروجی مرحلهٔ اول را متصل کنید و مسیر `data/pilot_v1` را در متغیر `DATA_ROOT` مرحلهٔ 12 بنویسید.
۴. اگر در همین جلسه ادامه می‌دهید، داده در مسیر فعلی موجود است.

ذخیرهٔ سورس در GitHub با ذخیرهٔ وزن/داده در خروجی Kaggle دو کار جداست. فایل‌های بزرگ را وارد Git نکنید.

In [ ]:
environment = {"git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "data_root": str(DATA_ROOT), "layout": LAYOUT, "next_notebook": "12_surt_streaming_inference.ipynb"}
(ARTIFACTS / "session.json").write_text(json.dumps(environment, indent=2))
(ARTIFACTS / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Save this directory as notebook output:", ARTIFACTS)